# 🧹 NYC Airbnb Data Cleaning & Reporting Automation

## Automated Data Cleaning Pipeline

**Project:** Automated Data Cleaning & Reporting System  
**Dataset:** NYC Airbnb Open Data  
**Input File:** `AB_NYC_2019.csv`  
**Output File:** `AB_NYC_2019_cleaned.csv`

---

### Objective

This notebook implements an automated data-cleaning pipeline for the NYC Airbnb dataset.

The cleaning process addresses:

- Missing values
- Duplicate records
- Duplicate listing IDs
- Incorrect data types
- Inconsistent text formatting
- Invalid numerical values
- Extreme values and outliers

The cleaned dataset will be validated and exported for further analysis and automated reporting.

> **The original raw dataset is never modified.**

## 🔄 Cleaning Workflow

The automated cleaning process follows these stages:

1. Load the raw dataset
2. Create a backup copy
3. Generate the pre-cleaning quality baseline
4. Handle missing values
5. Remove duplicate records
6. Correct data types
7. Standardize categorical and text data
8. Handle invalid numerical values
9. Identify and treat extreme values
10. Validate the cleaned dataset
11. Generate a before-vs-after quality comparison
12. Export the cleaned dataset

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the original raw dataset

df = pd.read_csv("../data/raw/AB_NYC_2019.csv")

print("Raw dataset loaded successfully!")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Raw dataset loaded successfully!
Rows: 48,895
Columns: 16


In [3]:
# Create a working copy so the original raw data remains untouched

cleaned_df = df.copy()

print("Working copy created successfully.")

Working copy created successfully.


## 📋 Step 1 — Pre-Cleaning Data Quality Baseline

Before making any changes, we record the quality of the raw dataset.

This baseline will be compared with the dataset after cleaning to measure the effectiveness of the automated cleaning process.

In [4]:
# Calculate data-quality metrics before cleaning

before_metrics = {
    "Rows": len(cleaned_df),
    "Columns": len(cleaned_df.columns),
    "Missing Values": cleaned_df.isnull().sum().sum(),
    "Duplicate Rows": cleaned_df.duplicated().sum(),
    "Duplicate IDs": cleaned_df["id"].duplicated().sum(),
    "Zero Prices": (cleaned_df["price"] == 0).sum(),
    "Negative Prices": (cleaned_df["price"] < 0).sum(),
    "Invalid Minimum Nights": (cleaned_df["minimum_nights"] <= 0).sum(),
    "Invalid Availability": (
        (cleaned_df["availability_365"] < 0) |
        (cleaned_df["availability_365"] > 365)
    ).sum(),
    "Negative Review Counts": (
        cleaned_df["number_of_reviews"] < 0
    ).sum()
}

before_metrics

{'Rows': 48895,
 'Columns': 16,
 'Missing Values': np.int64(20141),
 'Duplicate Rows': np.int64(0),
 'Duplicate IDs': np.int64(0),
 'Zero Prices': np.int64(11),
 'Negative Prices': np.int64(0),
 'Invalid Minimum Nights': np.int64(0),
 'Invalid Availability': np.int64(0),
 'Negative Review Counts': np.int64(0)}

## 🕳️ Step 2 — Missing Value Handling

Missing values are handled according to the meaning of each column rather than applying a single rule to every column.

### Cleaning Strategy

- `name` → fill missing values with `"Unknown Listing"`
- `host_name` → fill missing values with `"Unknown Host"`
- `reviews_per_month` → fill missing values with `0`
- `last_review` → preserve missing values because the absence of a review date can represent a listing with no recorded review

This approach avoids unnecessary deletion of valid Airbnb listings.

In [5]:
# Display missing values before treatment

missing_before = cleaned_df.isnull().sum()

missing_before[missing_before > 0]

name                    16
host_name               21
last_review          10052
reviews_per_month    10052
dtype: int64

In [6]:
# Handle missing values using column-specific rules

cleaned_df["name"] = cleaned_df["name"].fillna("Unknown Listing")

cleaned_df["host_name"] = cleaned_df["host_name"].fillna("Unknown Host")

cleaned_df["reviews_per_month"] = cleaned_df["reviews_per_month"].fillna(0)

print("Missing-value treatment completed.")

Missing-value treatment completed.


In [7]:
# Check remaining missing values

missing_after = cleaned_df.isnull().sum()

remaining_missing = missing_after[missing_after > 0]

remaining_missing

last_review    10052
dtype: int64

## ♻️ Step 3 — Duplicate Removal

Duplicate records can cause inaccurate listing counts and distorted analysis.

The cleaning pipeline removes:

1. Completely duplicated rows
2. Duplicate Airbnb listing IDs

The `id` field is treated as the unique identifier for an Airbnb listing.

In [8]:
# Count duplicate records before removal

duplicate_rows_before = cleaned_df.duplicated().sum()
duplicate_ids_before = cleaned_df["id"].duplicated().sum()

print(f"Duplicate Rows: {duplicate_rows_before}")
print(f"Duplicate IDs: {duplicate_ids_before}")

Duplicate Rows: 0
Duplicate IDs: 0


In [9]:
# Remove complete duplicate rows

cleaned_df = cleaned_df.drop_duplicates()

# Remove duplicate listing IDs while keeping the first occurrence

cleaned_df = cleaned_df.drop_duplicates(
    subset="id",
    keep="first"
)

print("Duplicate removal completed.")
print(f"Remaining rows: {len(cleaned_df):,}")

Duplicate removal completed.
Remaining rows: 48,895


In [10]:
# Verify that duplicates have been removed

print("Duplicate Rows After Cleaning:", cleaned_df.duplicated().sum())
print("Duplicate IDs After Cleaning:", cleaned_df["id"].duplicated().sum())

Duplicate Rows After Cleaning: 0
Duplicate IDs After Cleaning: 0


## 🔤 Step 4 — Data Type Correction

Correct data types are required for reliable analysis and reporting.

The following conversions will be applied:

- `last_review` → datetime
- Numeric fields → appropriate numeric types

Date conversion is particularly important because it allows the dataset to support time-based analysis later.

In [11]:
# Convert last_review to datetime

cleaned_df["last_review"] = pd.to_datetime(
    cleaned_df["last_review"],
    errors="coerce"
)

print("Date conversion completed.")
print(cleaned_df["last_review"].dtype)

Date conversion completed.
datetime64[us]


In [12]:
# Ensure important numerical columns are numeric

numeric_columns = [
    "id",
    "host_id",
    "latitude",
    "longitude",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365"
]

for column in numeric_columns:
    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column],
        errors="coerce"
    )

print("Numeric data types standardized.")

Numeric data types standardized.


## 🏷️ Step 5 — Text & Category Standardization

Text inconsistencies can create duplicate categories during analysis.

The cleaning process will:

- Remove unnecessary leading/trailing whitespace
- Standardize text formatting
- Preserve the original meaning of categorical values

Important fields include:

- `room_type`
- `neighbourhood_group`
- `neighbourhood`

In [13]:
# Standardize text-based columns

text_columns = [
    "name",
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for column in text_columns:
    cleaned_df[column] = cleaned_df[column].astype("string").str.strip()

print("Text standardization completed.")

Text standardization completed.


In [14]:
# Display standardized room types

cleaned_df["room_type"].value_counts()

room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: Int64

In [15]:
# Display standardized neighbourhood groups

cleaned_df["neighbourhood_group"].value_counts()

neighbourhood_group
Manhattan        21661
Brooklyn         20104
Queens            5666
Bronx             1091
Staten Island      373
Name: count, dtype: Int64

## ⚠️ Step 6 — Invalid Numerical Value Handling

Logical validation rules are applied to numerical fields.

### Validation Rules

- Price must be greater than zero.
- Minimum nights must be greater than zero.
- Availability must be between 0 and 365 days.
- Number of reviews cannot be negative.
- Reviews per month cannot be negative.

Records violating these rules will be removed because they cannot represent valid values for the corresponding fields.

In [16]:
# Detect invalid numerical records

invalid_price = cleaned_df["price"] <= 0

invalid_nights = cleaned_df["minimum_nights"] <= 0

invalid_availability = (
    (cleaned_df["availability_365"] < 0) |
    (cleaned_df["availability_365"] > 365)
)

invalid_reviews = cleaned_df["number_of_reviews"] < 0

invalid_reviews_per_month = cleaned_df["reviews_per_month"] < 0

print("Invalid price records:", invalid_price.sum())
print("Invalid minimum-night records:", invalid_nights.sum())
print("Invalid availability records:", invalid_availability.sum())
print("Invalid review-count records:", invalid_reviews.sum())
print("Invalid reviews/month records:", invalid_reviews_per_month.sum())

Invalid price records: 11
Invalid minimum-night records: 0
Invalid availability records: 0
Invalid review-count records: 0
Invalid reviews/month records: 0


In [17]:
# Remove records containing logically invalid numerical values

invalid_mask = (
    invalid_price |
    invalid_nights |
    invalid_availability |
    invalid_reviews |
    invalid_reviews_per_month
)

invalid_records_removed = invalid_mask.sum()

cleaned_df = cleaned_df.loc[~invalid_mask].copy()

print(f"Invalid records removed: {invalid_records_removed}")
print(f"Remaining rows: {len(cleaned_df):,}")

Invalid records removed: 11
Remaining rows: 48,884


## 📦 Step 7 — Outlier Investigation

Extreme values are not automatically treated as errors.

For example, an Airbnb listing with a very high price may be unusual but could still represent a legitimate listing.

Therefore, the project will:

1. Identify extreme price values.
2. Measure their impact.
3. Avoid blindly deleting legitimate observations.
4. Apply business rules only where values are clearly unreasonable.

For this project, the price distribution will be investigated using the IQR method.

In [18]:
# Calculate IQR-based price boundaries

Q1 = cleaned_df["price"].quantile(0.25)
Q3 = cleaned_df["price"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: ${Q1:,.2f}")
print(f"Q3: ${Q3:,.2f}")
print(f"IQR: ${IQR:,.2f}")
print(f"Lower Bound: ${lower_bound:,.2f}")
print(f"Upper Bound: ${upper_bound:,.2f}")

Q1: $69.00
Q3: $175.00
IQR: $106.00
Lower Bound: $-90.00
Upper Bound: $334.00


In [19]:
# Count price observations outside the IQR boundaries

price_outliers = (
    (cleaned_df["price"] < lower_bound) |
    (cleaned_df["price"] > upper_bound)
)

print(f"Potential price outliers: {price_outliers.sum():,}")
print(f"Percentage of data: {price_outliers.mean() * 100:.2f}%")

Potential price outliers: 2,972
Percentage of data: 6.08%


### Outlier Treatment Decision

Price outliers are retained when they represent potentially legitimate Airbnb listings.

This prevents the automated pipeline from deleting valid business observations simply because they are statistically unusual.

Clearly invalid values, such as zero or negative prices, were already removed through logical validation.

## 🧪 Step 8 — Data Validation After Cleaning

The cleaned dataset is now tested against the same quality rules used during the initial profiling.

The goal is to verify that:

- Duplicate records have been removed.
- Duplicate listing IDs are eliminated.
- Invalid prices are removed.
- Invalid minimum nights are removed.
- Invalid availability values are removed.
- Negative review values are removed.
- Required text fields have been handled.
- Data types are appropriate.

In [20]:
# Validate the cleaned dataset

validation_results = {
    "Duplicate Rows": cleaned_df.duplicated().sum(),
    "Duplicate IDs": cleaned_df["id"].duplicated().sum(),
    "Invalid Prices": (cleaned_df["price"] <= 0).sum(),
    "Invalid Minimum Nights": (cleaned_df["minimum_nights"] <= 0).sum(),
    "Invalid Availability": (
        (cleaned_df["availability_365"] < 0) |
        (cleaned_df["availability_365"] > 365)
    ).sum(),
    "Negative Review Counts": (
        cleaned_df["number_of_reviews"] < 0
    ).sum(),
    "Negative Reviews per Month": (
        cleaned_df["reviews_per_month"] < 0
    ).sum()
}

validation_results

{'Duplicate Rows': np.int64(0),
 'Duplicate IDs': np.int64(0),
 'Invalid Prices': np.int64(0),
 'Invalid Minimum Nights': np.int64(0),
 'Invalid Availability': np.int64(0),
 'Negative Review Counts': np.int64(0),
 'Negative Reviews per Month': np.int64(0)}

In [21]:
# Convert validation results into a table

validation_df = pd.DataFrame(
    list(validation_results.items()),
    columns=["Validation Check", "Remaining Issues"]
)

validation_df

,Validation Check,Remaining Issues
0,Duplicate Rows,0
1,Duplicate IDs,0
2,Invalid Prices,0
3,Invalid Minimum Nights,0
4,Invalid Availability,0
5,Negative Review Counts,0
6,Negative Reviews per Month,0


## 📊 Step 9 — Before vs After Data Quality Comparison

The final quality comparison measures the improvement produced by the cleaning pipeline.

This comparison will later be included in the automated Excel report.

In [22]:
# Calculate data-quality metrics after cleaning

after_metrics = {
    "Rows": len(cleaned_df),
    "Columns": len(cleaned_df.columns),
    "Missing Values": cleaned_df.isnull().sum().sum(),
    "Duplicate Rows": cleaned_df.duplicated().sum(),
    "Duplicate IDs": cleaned_df["id"].duplicated().sum(),
    "Zero Prices": (cleaned_df["price"] == 0).sum(),
    "Negative Prices": (cleaned_df["price"] < 0).sum(),
    "Invalid Minimum Nights": (cleaned_df["minimum_nights"] <= 0).sum(),
    "Invalid Availability": (
        (cleaned_df["availability_365"] < 0) |
        (cleaned_df["availability_365"] > 365)
    ).sum(),
    "Negative Review Counts": (
        cleaned_df["number_of_reviews"] < 0
    ).sum()
}

after_metrics

{'Rows': 48884,
 'Columns': 16,
 'Missing Values': np.int64(10051),
 'Duplicate Rows': np.int64(0),
 'Duplicate IDs': np.int64(0),
 'Zero Prices': np.int64(0),
 'Negative Prices': np.int64(0),
 'Invalid Minimum Nights': np.int64(0),
 'Invalid Availability': np.int64(0),
 'Negative Review Counts': np.int64(0)}

In [23]:
# Create before-vs-after comparison

comparison_df = pd.DataFrame({
    "Metric": before_metrics.keys(),
    "Before Cleaning": before_metrics.values(),
    "After Cleaning": after_metrics.values()
})

comparison_df

,Metric,Before Cleaning,After Cleaning
0,Rows,48895,48884
1,Columns,16,16
2,Missing Values,20141,10051
3,Duplicate Rows,0,0
4,Duplicate IDs,0,0
5,Zero Prices,11,0
6,Negative Prices,0,0
7,Invalid Minimum Nights,0,0
8,Invalid Availability,0,0
9,Negative Review Counts,0,0


In [24]:
# Calculate cleaning impact

records_removed = len(df) - len(cleaned_df)

print(f"Original Records: {len(df):,}")
print(f"Cleaned Records: {len(cleaned_df):,}")
print(f"Records Removed: {records_removed:,}")
print(
    f"Percentage Removed: "
    f"{records_removed / len(df) * 100:.2f}%"
)

Original Records: 48,895
Cleaned Records: 48,884
Records Removed: 11
Percentage Removed: 0.02%


## 💾 Step 10 — Export Cleaned Dataset

The cleaned dataset is exported separately from the raw dataset.

### Input

`data/raw/AB_NYC_2019.csv`

### Output

`data/cleaned/AB_NYC_2019_cleaned.csv`

The original raw dataset remains unchanged, allowing the cleaning process to be reproduced at any time.

In [25]:
# Save the cleaned dataset

output_path = "../data/cleaned/AB_NYC_2019_cleaned.csv"

cleaned_df.to_csv(
    output_path,
    index=False
)

print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: ../data/cleaned/AB_NYC_2019_cleaned.csv


## ✅ Data Cleaning Summary

The automated cleaning pipeline has completed the following operations:

- Loaded the raw Airbnb dataset.
- Preserved the original raw data.
- Identified and handled appropriate missing values.
- Removed duplicate records and duplicate listing IDs.
- Corrected important data types.
- Standardized text fields.
- Detected and removed logically invalid numerical records.
- Investigated price outliers using the IQR method.
- Validated the cleaned dataset.
- Compared data quality before and after cleaning.
- Exported the cleaned dataset.

### Next Phase

The cleaned dataset will be used for:

1. Exploratory analysis
2. KPI generation
3. Automated visualizations
4. Business insights
5. Automated Excel reporting